In [ ]:
# Problema: Comparar agregaciones parciales por partición con una agregación global, usando telemetría real y sin infraestructura distribuida.

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data").is_dir() and (p / "submission").is_dir()
)


In [ ]:
events = pd.read_csv(ROOT / "data/truck_events.csv.gz")
events.shape, events.eventType.value_counts()


In [ ]:
# En una plataforma distribuida estas particiones vivirían en trabajadores distintos.

partitions = list(np.array_split(events, 4))
[len(part) for part in partitions]


In [ ]:
# Cada partición produce un agregado parcial; después reducimos esos resultados.

partial = [
    part.groupby("eventType", as_index=False)
    .size()
    .rename(columns={"size": "event_count"})
    for part in partitions
]
result = (
    pd.concat(partial)
    .groupby("eventType", as_index=False)
    .event_count.sum()
    .sort_values("eventType")
)
result


In [ ]:
reference = (
    events.groupby("eventType", as_index=False)
    .size()
    .rename(columns={"size": "event_count"})
    .sort_values("eventType")
)
assert result.reset_index(drop=True).equals(reference.reset_index(drop=True))
result.to_parquet(ROOT / "submission/event_counts.parquet", index=False)
pd.DataFrame(
    [
        ["input_rows", len(events)],
        ["teaching_input_partitions", len(partitions)],
        ["shuffle_partitions", len(partitions)],
        ["output_event_types", len(result)],
        ["validation_status", "PASS"],
    ],
    columns=["metric", "value"],
).to_csv(ROOT / "submission/execution_summary.csv", index=False)
